# Azure Digital Twins Model Migration

This notebook demonstrates how to use the `adt_model_migrator` module to migrate DTDL models from a folder to your Azure Digital Twins instance.

## Prerequisites

1. Azure Digital Twins instance created
2. Authentication configured (Azure CLI, Service Principal, or Managed Identity)
3. DTDL models in the `Source/DTDLv2` folder

## Setup

First, import the migration module and set your Azure Digital Twins instance URL.

In [1]:
import sys
from pathlib import Path

# Add parent directory to path to import the module
sys.path.insert(0, str(Path.cwd().parent))

import adt_model_migrator

In [5]:
# Configure your Azure Digital Twins instance URL
# Format: https://your-instance-name.api.region.digitaltwins.azure.net
ADT_URL = "https://adt-dev-env.api.weu.digitaltwins.azure.net/"

# Path to your DTDL models folder
MODELS_FOLDER = "../Source/DTDLv2"

## Dry Run

Before actually migrating, you can do a dry run to see what would be uploaded without making any changes.

In [7]:
# Dry run - validates and prepares models without uploading
result = adt_model_migrator.migrate_models(
    folder_path=MODELS_FOLDER,
    adt_url=ADT_URL,
    delete_existing=False,  # Don't delete in dry run
    dry_run=True
)

print(f"Dry run result: {result}")

2026-01-16 12:23:58,864 - adt_model_migrator - INFO - Starting model migration from ../Source/DTDLv2
2026-01-16 12:23:58,865 - adt_model_migrator - INFO - Azure Digital Twins URL: https://adt-dev-env.api.weu.digitaltwins.azure.net/
2026-01-16 12:23:58,865 - adt_model_migrator - INFO - Delete existing: False, Dry run: True
2026-01-16 12:23:58,966 - adt_model_migrator - INFO - Found 1374 JSON files in ../Source/DTDLv2
2026-01-16 12:24:01,454 - adt_model_migrator - INFO - Loaded 1374 Interface models
2026-01-16 12:24:01,455 - adt_model_migrator - INFO - Loaded 1374 models
2026-01-16 12:24:01,459 - adt_model_migrator - INFO - Resolved dependencies, 1374 models ready for upload
2026-01-16 12:24:01,459 - adt_model_migrator - INFO - DRY RUN: Would upload the following models:
2026-01-16 12:24:01,460 - adt_model_migrator - INFO -   1. dtmi:org:w3id:rec:Collection;1
2026-01-16 12:24:01,460 - adt_model_migrator - INFO -   2. dtmi:org:w3id:rec:Portfolio;1
2026-01-16 12:24:01,460 - adt_model_migra

Dry run result: {'success': True, 'dry_run': True, 'loaded': 1374, 'ordered': 1374, 'warnings': []}


## Full Migration

This will:
1. Load all DTDL models from the folder
2. Validate the models
3. Resolve dependencies and order models correctly
4. Delete all existing models (if `delete_existing=True`)
   - **If models are in use by digital twins, those twins will be deleted to allow model deletion**
   - This ensures all models are deleted, even if they're in use
5. Upload all models in the correct dependency order

**Important:** When you re-upload a model with the same DTMI, any digital twins that were using that model will need to be recreated. The twins will then reference the updated model automatically. This matches ADT Explorer's "Delete All Models" behavior.

In [ ]:
# Full migration - delete all existing models and upload new models
# Note: If models are in use by digital twins, those twins will be deleted to allow model deletion
# After re-uploading, twins can be recreated and will use the updated models
result = adt_model_migrator.migrate_models(
    folder_path=MODELS_FOLDER,
    adt_url=ADT_URL,
    delete_existing=True,  # Delete all existing models (twins using them will be deleted too)
    dry_run=False
)

print("\n=== Migration Results ===")
print(f"Success: {result['success']}")
print(f"Models loaded: {result['loaded']}")
print(f"Models uploaded: {result['uploaded']}")
print(f"Models failed: {result.get('failed', 0)}")
print(f"Models deleted: {result.get('deleted', 0)}")
print("\nNote: All models were deleted. If any models were in use by digital twins,")
print("those twins were deleted to allow model deletion. You can now recreate")
print("the twins and they will use the updated models.")

if result.get('warnings'):
    print(f"\nWarnings: {len(result['warnings'])}")
    for warning in result['warnings'][:5]:  # Show first 5 warnings
        print(f"  - {warning}")

if result.get('upload_errors'):
    print(f"\nUpload Errors: {len(result['upload_errors'])}")
    for error in result['upload_errors'][:5]:  # Show first 5 errors
        print(f"  - {error}")

2026-01-16 12:24:57,165 - adt_model_migrator - INFO - Starting model migration from ../Source/DTDLv2
2026-01-16 12:24:57,166 - adt_model_migrator - INFO - Azure Digital Twins URL: https://adt-dev-env.api.weu.digitaltwins.azure.net/
2026-01-16 12:24:57,166 - adt_model_migrator - INFO - Delete existing: True, Dry run: False
2026-01-16 12:24:57,264 - adt_model_migrator - INFO - Found 1374 JSON files in ../Source/DTDLv2
2026-01-16 12:24:57,443 - adt_model_migrator - INFO - Loaded 1374 Interface models
2026-01-16 12:24:57,444 - adt_model_migrator - INFO - Loaded 1374 models
2026-01-16 12:24:57,447 - adt_model_migrator - INFO - Resolved dependencies, 1374 models ready for upload
2026-01-16 12:24:57,447 - azure.identity._credentials.environment - INFO - No environment configuration found.
/Users/sigve/Documents/code/rec/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8


=== Migration Results ===
Success: False
Models loaded: 1374
Models uploaded: 0
Models failed: 1374
Models deleted: 1053

Upload Errors: 14
  - Batch 1 failed with client error: (ModelIdAlreadyExists) Some of the model ids already exist: dtmi:org:brickschema:schema:Brick:Point;1, dtmi:org:brickschema:schema:Brick:Parameter;1, dtmi:org:w3id:rec:Asset;1, dtmi:org:brickschema:schema:Brick:Equipment;1, dtmi:org:w3id:rec:Collection;1, dtmi:org:w3id:rec:Agent;1, dtmi:org:w3id:rec:Information;1, dtmi:org:w3id:rec:ArchitectureArea;1, dtmi:org:w3id:rec:ArchitectureCapacity;1, dtmi:org:w3id:rec:Organization;1, dtmi:org:w3id:rec:ArchitecturalAsset;1, dtmi:org:w3id:rec:BarrierAsset;1, dtmi:org:w3id:rec:ICTEquipment;1, dtmi:org:w3id:rec:SensorEquipment;1, dtmi:org:w3id:rec:Controller;1, dtmi:org:w3id:rec:Furniture;1, dtmi:org:w3id:rec:Stand;1, dtmi:org:w3id:rec:Table;1, dtmi:org:w3id:rec:Lamp;1, dtmi:org:w3id:rec:ServiceObject;1. Use Model_List API to view models that already exist. See the Swagge

## Upload Without Deleting

If you want to add new models without deleting existing ones, set `delete_existing=False`. Note that this may fail if models with the same ID already exist.

## Advanced Usage

You can also use the individual functions for more control:

In [ ]:
# Load and validate models manually
from pathlib import Path

models = adt_model_migrator.load_dtdl_models(Path(MODELS_FOLDER))
print(f"Loaded {len(models)} models")

# Resolve dependencies
ordered_models, warnings = adt_model_migrator.resolve_dependencies(models)
print(f"Ordered {len(ordered_models)} models")
if warnings:
    print(f"Warnings: {warnings}")

# Validate models
is_valid, errors = adt_model_migrator.validate_models(ordered_models)
print(f"Validation: {is_valid}")
if errors:
    print(f"Errors: {errors}")

## Troubleshooting

### Authentication Issues

If you encounter authentication errors, try:

1. **Azure CLI**: Run `az login` in your terminal
2. **Service Principal**: Set environment variables:
   - `AZURE_CLIENT_ID`
   - `AZURE_CLIENT_SECRET`
   - `AZURE_TENANT_ID`
3. **Managed Identity**: Ensure you're running in an Azure environment with managed identity enabled

### Model Upload Failures

Common issues:
- **Missing dependencies**: Ensure all models that are extended or referenced are included
- **Invalid DTMI format**: Check that all `@id` fields follow the `dtmi:...` format
- **Duplicate models**: Ensure no duplicate model IDs in your folder
- **Models in use**: If deleting fails, some models may be in use by digital twins

## Debugging Upload Errors

If you're getting upload errors, use this cell to get more detailed information about which models are failing and why.

In [ ]:
# Debug specific models that failed
from azure.identity import DefaultAzureCredential
from azure.digitaltwins.core import DigitalTwinsClient
from pathlib import Path

# Load models
models = adt_model_migrator.load_dtdl_models(Path(MODELS_FOLDER))
ordered_models, warnings = adt_model_migrator.resolve_dependencies(models)

print(f"Total models: {len(ordered_models)}")
if warnings:
    print(f"\nWarnings: {len(warnings)}")
    for w in warnings[:10]:
        print(f"  - {w}")

# Test uploading a small batch to see detailed errors
credential = DefaultAzureCredential()
client = DigitalTwinsClient(ADT_URL.rstrip('/'), credential)

# Try uploading first 5 models to see if there are immediate issues
test_batch = ordered_models[:5]
print(f"\nTesting upload of first 5 models:")
for i, model in enumerate(test_batch, 1):
    model_id = model.get("@id", "unknown")
    print(f"\n{i}. Testing {model_id}...")
    try:
        client.create_models([model])
        print(f"   ✓ Success")
    except Exception as e:
        print(f"   ✗ Failed: {e}")
        # Try to get more details
        if hasattr(e, 'response'):
            try:
                if hasattr(e.response, 'text'):
                    print(f"   Response: {e.response.text[:500]}")
            except:
                pass

## Check Existing Models and Digital Twins

If models can't be deleted because they're in use, you may need to delete the digital twins first. Use this cell to check what's in your ADT instance.

In [ ]:
# Check existing models and digital twins
from azure.identity import DefaultAzureCredential
from azure.digitaltwins.core import DigitalTwinsClient

credential = DefaultAzureCredential()
client = DigitalTwinsClient(ADT_URL.rstrip('/'), credential)

# List all models
print("=== Existing Models ===")
models = list(client.list_models())
print(f"Total models: {len(models)}")
for i, model in enumerate(models[:10], 1):
    print(f"  {i}. {model.id}")
if len(models) > 10:
    print(f"  ... and {len(models) - 10} more")

# List all digital twins (if any)
print("\n=== Digital Twins ===")
try:
    twins = list(client.list_digital_twins())
    print(f"Total digital twins: {len(twins)}")
    if twins:
        print("WARNING: Digital twins exist! Models in use by these twins cannot be deleted.")
        print("You may need to delete the digital twins first before deleting models.")
        for i, twin in enumerate(twins[:10], 1):
            print(f"  {i}. {twin.get('$dtId', 'unknown')} (model: {twin.get('$metadata', {}).get('$model', 'unknown')})")
        if len(twins) > 10:
            print(f"  ... and {len(twins) - 10} more")
    else:
        print("No digital twins found. Models should be deletable.")
except Exception as e:
    print(f"Error listing digital twins: {e}")
    print("(This might be a permissions issue)")